# R-CNN Object Detection — Complete Implementation

This notebook implements the original **R-CNN (Region-based Convolutional Neural Network)** pipeline on the PASCAL VOC 2012 dataset.

### Pipeline Overview
1. **Dataset** — PASCAL VOC 2012 (20 classes)
2. **Step 1** — Region Proposal Generation via Selective Search (~2000 per image)
3. **Step 2** — Feature Extraction using pretrained VGG16 (penultimate layer)
4. **Step 3** — Train Linear SVM classifier per class (IoU threshold ≥ 0.5)
5. **Step 4** — Bounding Box Regression refinement
6. **Step 5** — Testing + Non-Maximum Suppression (NMS)
7. **Evaluation** — IoU, mAP (with/without regression); Compare R-CNN vs Faster R-CNN

## 0. Install & Import Dependencies

In [ ]:
!pip install -q torch torchvision opencv-python matplotlib scikit-learn selectivesearch tqdm

In [ ]:
import os, random, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import selectivesearch
from tqdm import tqdm
from PIL import Image
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from torchvision.datasets import VOCDetection

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Dataset — PASCAL VOC 2012

In [ ]:
DATA_ROOT = './data'
os.makedirs(DATA_ROOT, exist_ok=True)

# VOC 2012 classes
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog',
    'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa',
    'train', 'tvmonitor'
]
NUM_CLASSES = len(VOC_CLASSES)  # 21 (including background)
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

# Load dataset (images as tensors, targets as XML-parsed dicts)
transform = T.ToTensor()
train_dataset = VOCDetection(root=DATA_ROOT, year='2012',
                              image_set='train', download=True,
                              transform=transform)
val_dataset   = VOCDetection(root=DATA_ROOT, year='2012',
                              image_set='val', download=True,
                              transform=transform)

print(f'Train images: {len(train_dataset)}')
print(f'Val   images: {len(val_dataset)}')

In [ ]:
def parse_annotations(target):
    """Extract list of (class_name, [x1,y1,x2,y2]) from VOC XML target."""
    boxes = []
    objs = target['annotation']['object']
    if isinstance(objs, dict):  # single object
        objs = [objs]
    for obj in objs:
        name = obj['name']
        bb   = obj['bndbox']
        x1, y1 = int(bb['xmin']), int(bb['ymin'])
        x2, y2 = int(bb['xmax']), int(bb['ymax'])
        boxes.append((name, [x1, y1, x2, y2]))
    return boxes

# Quick check
img0, tgt0 = train_dataset[0]
anns0 = parse_annotations(tgt0)
print('Sample annotations:', anns0[:3])

## Step 1 — Region Proposal Generation (Selective Search)

In [ ]:
def get_proposals(img_tensor, max_proposals=2000):
    """
    Run Selective Search on a CHW float tensor [0,1].
    Returns list of (x, y, w, h) rects.
    """
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    _, regions = selectivesearch.selective_search(
        img_np, scale=200, sigma=0.9, min_size=10
    )
    seen = set()
    proposals = []
    for r in regions:
        rect = r['rect']  # (x, y, w, h)
        if rect not in seen:
            seen.add(rect)
            proposals.append(list(rect))
        if len(proposals) >= max_proposals:
            break
    return proposals  # [[x,y,w,h], ...]

print('Selective Search function defined.')

In [ ]:
# Visualize top-20 proposals on first image
img_t, tgt = train_dataset[0]
proposals   = get_proposals(img_t)
print(f'Total proposals generated: {len(proposals)}')

img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Original with GT boxes
axes[0].imshow(img_np)
axes[0].set_title('Ground-Truth Boxes')
for name, (x1, y1, x2, y2) in parse_annotations(tgt):
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                          linewidth=2, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-5, name, color='lime', fontsize=9)

# Top-20 proposals
axes[1].imshow(img_np)
axes[1].set_title('Top-20 Region Proposals (Selective Search)')
colors = plt.cm.hsv(np.linspace(0, 1, 20))
for i, (x, y, w, h) in enumerate(proposals[:20]):
    rect = plt.Rectangle((x, y), w, h, linewidth=1.5,
                          edgecolor=colors[i], facecolor='none')
    axes[1].add_patch(rect)

plt.tight_layout()
plt.savefig('region_proposals.png', dpi=100)
plt.show()
print('Visualization saved to region_proposals.png')

## Step 2 — Feature Extraction using Pretrained VGG16

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

# Load VGG16 pretrained on ImageNet
vgg = vgg16(weights=VGG16_Weights.DEFAULT)

# Remove the final classification layer → extract 4096-d features
# features: conv layers; classifier[:6] = fc6 + relu + dropout + fc7 + relu + dropout
feature_extractor = nn.Sequential(
    vgg.features,
    vgg.avgpool,
    nn.Flatten(),
    *list(vgg.classifier.children())[:-1]  # drop last Linear(4096→1000)
).to(DEVICE).eval()

# Verify output dimension
with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224).to(DEVICE)
    out   = feature_extractor(dummy)
    FEAT_DIM = out.shape[1]
print(f'Feature vector dimension: {FEAT_DIM}')  # 4096

In [ ]:
# Standard ImageNet normalisation
PREPROCESS = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def extract_features(img_tensor, boxes_xywh, batch_size=32):
    """
    Crop each proposal from img_tensor, resize to 224×224,
    and extract features with VGG16.

    img_tensor : (C, H, W) float [0,1]
    boxes_xywh : list of [x, y, w, h]
    Returns    : np.ndarray (N, 4096)
    """
    C, H, W = img_tensor.shape
    feats = []
    batch = []
    with torch.no_grad():
        for (x, y, w, h) in boxes_xywh:
            x1, y1 = max(x, 0), max(y, 0)
            x2, y2 = min(x+w, W), min(y+h, H)
            if x2 <= x1 or y2 <= y1:
                batch.append(torch.zeros(3, 224, 224))
            else:
                crop = img_tensor[:, y1:y2, x1:x2]
                crop = PREPROCESS(crop)
                batch.append(crop)
            if len(batch) == batch_size:
                inp  = torch.stack(batch).to(DEVICE)
                out  = feature_extractor(inp)
                feats.append(out.cpu().numpy())
                batch = []
        if batch:
            inp  = torch.stack(batch).to(DEVICE)
            out  = feature_extractor(inp)
            feats.append(out.cpu().numpy())
    return np.vstack(feats) if feats else np.empty((0, FEAT_DIM))

print('Feature extraction function defined.')

## IoU Utility

In [ ]:
def compute_iou(box_a, box_b):
    """
    Compute IoU between two boxes in [x1,y1,x2,y2] format.
    """
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union  = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x+w, y+h]

print('IoU helpers defined.')

## Step 3 — Build Training Dataset & Train Linear SVM

> **IoU ≥ 0.5** → positive sample for that class  
> **IoU < 0.3** → background (negative)  
> **0.3 ≤ IoU < 0.5** → ignored

In [ ]:
# ---------------------------------------------------------------
# CONFIG — reduce N_TRAIN_IMAGES for a quick demo on CPU
N_TRAIN_IMAGES = 100   # increase to 500+ for better accuracy
MAX_PROPOSALS  = 2000
IOU_POS        = 0.5
IOU_NEG        = 0.3
# ---------------------------------------------------------------

all_features = []   # list of np arrays
all_labels   = []   # int class index
all_regressions = []  # (dx, dy, dw, dh) for BBox reg
all_gt_boxes    = []  # matched GT box for each positive sample

print(f'Building training data from {N_TRAIN_IMAGES} images...')
for idx in tqdm(range(N_TRAIN_IMAGES), desc='Images'):
    img_t, tgt = train_dataset[idx]
    gt_annots   = parse_annotations(tgt)

    proposals = get_proposals(img_t, MAX_PROPOSALS)
    if not proposals:
        continue

    feats = extract_features(img_t, proposals)

    for i, (x, y, w, h) in enumerate(proposals):
        prop_xyxy = xywh_to_xyxy([x, y, w, h])
        best_iou  = 0.0
        best_cls  = 0        # background
        best_gt   = None

        for (cls_name, gt_box) in gt_annots:
            iou = compute_iou(prop_xyxy, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_cls  = CLASS_TO_IDX[cls_name]
                best_gt   = gt_box

        if best_iou >= IOU_POS:          # POSITIVE
            all_features.append(feats[i])
            all_labels.append(best_cls)
            # Regression targets
            px, py = x + w/2, y + h/2
            gx1, gy1, gx2, gy2 = best_gt
            gw, gh = gx2-gx1, gy2-gy1
            gcx, gcy = gx1+gw/2, gy1+gh/2
            tx = (gcx - px) / (w+1e-6)
            ty = (gcy - py) / (h+1e-6)
            tw = np.log((gw+1e-6) / (w+1e-6))
            th = np.log((gh+1e-6) / (h+1e-6))
            all_regressions.append([tx, ty, tw, th])
            all_gt_boxes.append(best_gt)

        elif best_iou < IOU_NEG:          # NEGATIVE (background)
            all_features.append(feats[i])
            all_labels.append(0)          # 0 = background
            all_regressions.append([0., 0., 0., 0.])
            all_gt_boxes.append([0, 0, 0, 0])
        # else: ignore

X = np.array(all_features,    dtype=np.float32)
y = np.array(all_labels,      dtype=np.int32)
R = np.array(all_regressions, dtype=np.float32)

pos_count = (y > 0).sum()
neg_count = (y == 0).sum()
print(f'\nTotal samples : {len(X)}')
print(f'  Positives   : {pos_count}')
print(f'  Negatives   : {neg_count}')
print(f'Feature shape : {X.shape}')

In [ ]:
from sklearn.model_selection import train_test_split

# Normalise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, R_train, R_test = train_test_split(
    X_scaled, y, R, test_size=0.2, random_state=42, stratify=np.clip(y, 0, 1)
)

print('Training One-vs-Rest Linear SVMs (one per class)...')
svms = {}  # class_idx → trained LinearSVC
for cls_idx in tqdm(range(1, NUM_CLASSES), desc='SVM Classes'):
    y_bin = (y_train == cls_idx).astype(int)
    if y_bin.sum() < 2:          # need at least 2 positives
        continue
    svm = LinearSVC(C=0.001, max_iter=2000)
    svm.fit(X_train, y_bin)
    svms[cls_idx] = svm

print(f'Trained SVMs for {len(svms)} classes.')

In [ ]:
from sklearn.metrics import classification_report

# Predict with multi-class approach: pick class with highest decision score
def predict_class(feats_scaled):
    if not svms:
        return np.zeros(len(feats_scaled), dtype=int)
    scores = np.zeros((len(feats_scaled), NUM_CLASSES))
    for cls_idx, svm in svms.items():
        scores[:, cls_idx] = svm.decision_function(feats_scaled)
    return np.argmax(scores, axis=1)

y_pred = predict_class(X_test)

# Show simple accuracy
acc = (y_pred == y_test).mean()
print(f'Test accuracy: {acc:.4f}')
print('\nClassification Report (top classes):')
unique_cls = sorted(set(y_test.tolist()))
class_names = [VOC_CLASSES[i] for i in unique_cls]
print(classification_report(y_test, y_pred, labels=unique_cls,
                             target_names=class_names, zero_division=0))

## Step 4 — Bounding Box Regression

In [ ]:
from sklearn.linear_model import Ridge

# Train a regression model per class (positives only)
regressors = {}
for cls_idx in tqdm(range(1, NUM_CLASSES), desc='BBox Regressors'):
    mask = (y_train == cls_idx)
    if mask.sum() < 2:
        continue
    X_pos = X_train[mask]
    R_pos = R_train[mask]
    reg = Ridge(alpha=1.0)
    reg.fit(X_pos, R_pos)
    regressors[cls_idx] = reg

print(f'Trained regressors for {len(regressors)} classes.')

def apply_regression(proposals_xywh, pred_deltas):
    """
    Apply predicted (tx,ty,tw,th) deltas to proposal boxes.
    proposals_xywh : (N, 4) [x, y, w, h]
    pred_deltas    : (N, 4) [tx, ty, tw, th]
    Returns refined [x1, y1, x2, y2] boxes.
    """
    refined = []
    for (x, y, w, h), (tx, ty, tw, th) in zip(proposals_xywh, pred_deltas):
        cx = x + w/2 + tx * w
        cy = y + h/2 + ty * h
        nw  = w * np.exp(tw)
        nh  = h * np.exp(th)
        refined.append([cx - nw/2, cy - nh/2, cx + nw/2, cy + nh/2])
    return np.array(refined)

print('BBox regression helpers defined.')

## Step 5 — Testing + Non-Maximum Suppression (NMS)

In [ ]:
def nms(boxes_xyxy, scores, iou_thresh=0.3):
    """
    Non-Maximum Suppression.
    boxes_xyxy : (N, 4)
    scores     : (N,)
    Returns indices of kept boxes.
    """
    if len(boxes_xyxy) == 0:
        return []
    order = np.argsort(scores)[::-1]
    kept  = []
    while len(order) > 0:
        i = order[0]
        kept.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([compute_iou(boxes_xyxy[i], boxes_xyxy[j]) for j in rest])
        order = rest[ious < iou_thresh]
    return kept

def detect(img_tensor, class_idx, conf_thresh=0.5, nms_thresh=0.3,
           use_regression=True):
    """
    Run full R-CNN detection pipeline for a single class on one image.
    Returns (kept_boxes, kept_scores) both as (N,4) / (N,) arrays.
    """
    proposals = get_proposals(img_tensor)
    if not proposals:
        return np.empty((0,4)), np.array([])

    feats = extract_features(img_tensor, proposals)
    feats_sc = scaler.transform(feats)

    if class_idx not in svms:
        return np.empty((0,4)), np.array([])

    raw_scores = svms[class_idx].decision_function(feats_sc)
    mask = raw_scores > conf_thresh
    if mask.sum() == 0:
        return np.empty((0,4)), np.array([])

    pos_props  = np.array(proposals)[mask]
    pos_scores = raw_scores[mask]
    pos_feats  = feats_sc[mask]

    # Bounding box refinement
    if use_regression and class_idx in regressors:
        deltas    = regressors[class_idx].predict(pos_feats)
        boxes_ref = apply_regression(pos_props, deltas)
    else:
        boxes_ref = np.array([xywh_to_xyxy(b) for b in pos_props])

    kept_idx = nms(boxes_ref, pos_scores, nms_thresh)
    return boxes_ref[kept_idx], pos_scores[kept_idx]

print('Detection (NMS) pipeline defined.')

In [ ]:
# Demo detection on the first validation image
img_demo, tgt_demo = val_dataset[0]
gt_anns = parse_annotations(tgt_demo)

# Pick first GT class
demo_cls_name, demo_gt_box = gt_anns[0]
demo_cls_idx  = CLASS_TO_IDX[demo_cls_name]

print(f'Running detection for class: {demo_cls_name} (idx={demo_cls_idx})')
boxes_r, scores_r = detect(img_demo, demo_cls_idx, conf_thresh=0.2, use_regression=True)
boxes_nr, scores_nr = detect(img_demo, demo_cls_idx, conf_thresh=0.2, use_regression=False)

img_vis = (img_demo.permute(1,2,0).numpy() * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title in zip(axes, ['Ground Truth', 'No Regression', 'With Regression']):
    ax.imshow(img_vis)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

# GT
for cname, (x1,y1,x2,y2) in gt_anns:
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-4, cname, color='lime', fontsize=9)

# No regression
for (x1,y1,x2,y2), s in zip(boxes_nr, scores_nr):
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x1, y1-4, f'{s:.2f}', color='red', fontsize=8)

# With regression
for (x1,y1,x2,y2), s in zip(boxes_r, scores_r):
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, edgecolor='cyan', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].text(x1, y1-4, f'{s:.2f}', color='cyan', fontsize=8)

plt.tight_layout()
plt.savefig('detection_result.png', dpi=100)
plt.show()

## Evaluation — IoU, mAP (with/without Regression)

In [ ]:
def compute_ap(recalls, precisions):
    """Compute VOC-style Average Precision using 11-point interpolation."""
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        prec = precisions[recalls >= t]
        ap  += prec.max() if prec.size > 0 else 0.0
    return ap / 11.0

def evaluate_class(dataset, class_idx, n_images=50, use_regression=True,
                   conf_thresh=0.0, nms_thresh=0.3, iou_thresh=0.5):
    """
    Evaluate mAP for a single class over n_images from dataset.
    Returns (AP, mean_iou_of_detections).
    """
    all_scores, all_tp, all_fp, n_gt = [], [], [], 0
    iou_list = []

    for idx in tqdm(range(n_images), desc=f'Eval {VOC_CLASSES[class_idx]}', leave=False):
        img_t, tgt = dataset[idx]
        gt_anns = [(c, b) for c, b in parse_annotations(tgt) if CLASS_TO_IDX[c] == class_idx]
        n_gt += len(gt_anns)

        boxes, scores = detect(img_t, class_idx,
                               conf_thresh=conf_thresh,
                               nms_thresh=nms_thresh,
                               use_regression=use_regression)
        matched = [False] * len(gt_anns)
        for box, score in zip(boxes, scores):
            all_scores.append(score)
            best_iou, best_j = 0.0, -1
            for j, (_, gt_box) in enumerate(gt_anns):
                iou = compute_iou(box, gt_box)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            iou_list.append(best_iou)
            if best_iou >= iou_thresh and best_j >= 0 and not matched[best_j]:
                all_tp.append(1); all_fp.append(0)
                matched[best_j] = True
            else:
                all_tp.append(0); all_fp.append(1)

    if not all_scores:
        return 0.0, 0.0

    order   = np.argsort(-np.array(all_scores))
    tp_cum  = np.cumsum(np.array(all_tp)[order])
    fp_cum  = np.cumsum(np.array(all_fp)[order])
    recalls = tp_cum / max(n_gt, 1)
    precs   = tp_cum / (tp_cum + fp_cum + 1e-9)
    ap      = compute_ap(recalls, precs)
    return ap, float(np.mean(iou_list)) if iou_list else 0.0

print('Evaluation helpers defined.')

In [ ]:
# Evaluate on a small subset for speed — increase N_EVAL for real experiments
N_EVAL = 30
results_noreg = {}   # class_idx → (AP, mIoU)
results_reg   = {}

eval_classes = [c for c in svms.keys()][:5]  # evaluate first 5 trained classes

for cls_idx in eval_classes:
    print(f'\nEvaluating class: {VOC_CLASSES[cls_idx]}')
    ap_nr, iou_nr = evaluate_class(val_dataset, cls_idx, n_images=N_EVAL,
                                    use_regression=False)
    ap_r,  iou_r  = evaluate_class(val_dataset, cls_idx, n_images=N_EVAL,
                                    use_regression=True)
    results_noreg[cls_idx] = (ap_nr, iou_nr)
    results_reg[cls_idx]   = (ap_r,  iou_r)
    print(f'  Without Reg → AP={ap_nr:.4f}  mIoU={iou_nr:.4f}')
    print(f'  With    Reg → AP={ap_r:.4f}  mIoU={iou_r:.4f}')

In [ ]:
# Summary bar chart
cls_names = [VOC_CLASSES[i] for i in eval_classes]
aps_nr = [results_noreg[i][0] for i in eval_classes]
aps_r  = [results_reg[i][0]   for i in eval_classes]

x = np.arange(len(cls_names))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, aps_nr, w, label='Without BBox Regression', color='steelblue')
ax.bar(x + w/2, aps_r,  w, label='With BBox Regression',    color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(cls_names, rotation=30, ha='right')
ax.set_ylabel('Average Precision (AP)')
ax.set_title('R-CNN: AP per Class — With vs Without BBox Regression')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('ap_comparison.png', dpi=100)
plt.show()

mAP_nr = np.mean(aps_nr)
mAP_r  = np.mean(aps_r)
print(f'mAP  Without Reg : {mAP_nr:.4f}')
print(f'mAP  With    Reg : {mAP_r:.4f}')

## Comparison — R-CNN vs Faster R-CNN

The table below uses established benchmarks; Faster R-CNN is loaded from
`torchvision.models.detection` and run on the same demo image for a side-by-side
qualitative comparison.

In [ ]:
import pandas as pd

data = {
    'Attribute'         : ['Backbone', 'Region Proposals', 'Proposal Speed',
                            'Feature Sharing', 'RPN (end-to-end)', 'Test Speed',
                            'VOC 2007 mAP'],
    'R-CNN'             : ['AlexNet/VGG', 'Selective Search (CPU)',
                            '~2s / image', 'No (crop → CNN per region)',
                            'No', '~47s / image', '~66%'],
    'Fast R-CNN'        : ['VGG16', 'Selective Search (CPU)',
                            '~2s / image', 'Yes (RoI pooling)',
                            'No', '~2s / image', '~70%'],
    'Faster R-CNN'      : ['VGG16 / ResNet', 'Region Proposal Network (GPU)',
                            '~0.01s / image', 'Yes (RPN + RoI pooling)',
                            'Yes', '~0.2s / image', '~73–78%'],
}
df = pd.DataFrame(data)
print(df.to_string(index=False))
df.style.set_caption('R-CNN Family Comparison')

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# COCO class names (Faster R-CNN pretrained on COCO)
COCO_CLASSES = [
    '__background__','person','bicycle','car','motorcycle','airplane','bus',
    'train','truck','boat','traffic light','fire hydrant','N/A','stop sign',
    'parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant',
    'bear','zebra','giraffe','N/A','backpack','umbrella','N/A','N/A','handbag',
    'tie','suitcase','frisbee','skis','snowboard','sports ball','kite',
    'baseball bat','baseball glove','skateboard','surfboard','tennis racket',
    'bottle','N/A','wine glass','cup','fork','knife','spoon','bowl','banana',
    'apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut',
    'cake','chair','couch','potted plant','bed','N/A','dining table','N/A',
    'N/A','toilet','N/A','tv','laptop','mouse','remote','keyboard','cell phone',
    'microwave','oven','toaster','sink','refrigerator','N/A','book','clock',
    'vase','scissors','teddy bear','hair drier','toothbrush'
]

faster_rcnn = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
).to(DEVICE).eval()

# Run Faster R-CNN on demo image
img_normalized = PREPROCESS(img_demo).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    preds = faster_rcnn([img_demo.to(DEVICE)])[0]

# Visualise side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax in axes:
    ax.imshow(img_vis)
    ax.axis('off')

# R-CNN detections (our model)
axes[0].set_title('R-CNN (our model)', fontsize=13)
for (x1,y1,x2,y2), s in zip(boxes_r, scores_r):
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, edgecolor='red', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-5, f'{demo_cls_name} {s:.2f}', color='red', fontsize=8)

# GT overlay
for cname, (x1,y1,x2,y2) in gt_anns:
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, linestyle='--', edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)

# Faster R-CNN detections
axes[1].set_title('Faster R-CNN (pretrained, COCO)', fontsize=13)
for box, lbl, score in zip(preds['boxes'], preds['labels'], preds['scores']):
    if score < 0.5:
        continue
    x1,y1,x2,y2 = box.cpu().numpy()
    cls_name = COCO_CLASSES[lbl.item()] if lbl < len(COCO_CLASSES) else str(lbl.item())
    rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                          lw=2, edgecolor='cyan', facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x1, y1-5, f'{cls_name} {score:.2f}', color='cyan', fontsize=8)

plt.tight_layout()
plt.savefig('rcnn_vs_faster_rcnn.png', dpi=100)
plt.show()
print('Comparison saved to rcnn_vs_faster_rcnn.png')

## Summary

| Step | What we did |
|------|-------------|
| Dataset | PASCAL VOC 2012 (20 object classes) |
| Step 1 | Selective Search → ~2000 region proposals / image; top-20 visualised |
| Step 2 | VGG16 (final FC removed) → 4096-d feature vector per region |
| Step 3 | One-vs-Rest Linear SVM; IoU ≥ 0.5 = positive, < 0.3 = negative |
| Step 4 | Ridge regression on positive features → refined (tx,ty,tw,th) deltas |
| Step 5 | SVM scoring + NMS (IoU < 0.3 suppressed) → final detections |
| Eval | IoU, AP & mAP computed with/without BBox regression |
| Compare | R-CNN vs Fast R-CNN vs Faster R-CNN (table + qualitative demo) |

**Key takeaway:** BBox regression consistently improves localisation (higher mIoU)
and typically raises AP. Faster R-CNN integrates region proposals into the network
(RPN), making it ~200× faster at test time while achieving higher mAP.